In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
from matplotlib import pyplot as plt
from tqdm import tqdm
import os
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
(RESULTS / 'extraction-rasters').mkdir(parents=True, exist_ok=True)

In [3]:
pseudo_codes_economies_142 = pd.read_csv(RESULTS / 'pseudo-codes-economies-142.csv')

In [4]:
def distribute_national_extraction_to_pixel(material, start, end, scenario, res='50km'):
    future = np.linspace(2023, 2050, 28)
    RESULTS = ROOT / "results"
    economy_raster_ = rio.open(RESULTS / 'economy-rasters' / f'economy-raster-eck4-{res}.tif')
    economy_raster = economy_raster_.read(1)

    RESULTS = Path(r'F:\aggregates-mining-biodiversity\results')
    input_path_1 = RESULTS / 'building-mineral-rasters' / f'global-building-minerals-{int(end-5)}-SSP2-{res}.tif'
    input_path_2 = RESULTS / 'building-mineral-rasters' / f'global-building-minerals-{end}-SSP2-{res}.tif'

    global_building_minerals_start = rio.open(input_path_1).read(1)
    global_building_minerals_end = rio.open(input_path_2).read(1)

    extraction_demand_raster_for_stock_growth_l = np.zeros(shape=economy_raster_.shape, dtype="float32")
    extraction_demand_raster_for_stock_growth_m = np.zeros(shape=economy_raster_.shape, dtype="float32")
    extraction_demand_raster_for_stock_growth_u = np.zeros(shape=economy_raster_.shape, dtype="float32")
    extraction_demand_raster_for_stock_turnover_l = np.zeros(shape=economy_raster_.shape, dtype="float32")
    extraction_demand_raster_for_stock_turnover_m = np.zeros(shape=economy_raster_.shape, dtype="float32")
    extraction_demand_raster_for_stock_turnover_u = np.zeros(shape=economy_raster_.shape, dtype="float32")

    with tqdm(range(1, 143), desc=f'distributing of {material} extraction demand in {start}-{end} of {scenario}') as pbar:
        for i in pbar:
            economy = pseudo_codes_economies_142[pseudo_codes_economies_142['value'] == i]['economies_142'].values[0]
            pbar.set_postfix_str(f'country={economy}')
            
            path_1 = RESULTS / 'material-flows' / economy / scenario / f'{material}_extraction_for_stock_turnover.npz'
            path_2 = RESULTS / 'material-flows' / economy / scenario / f'{material}_extraction_for_stock_growth.npz'

            value_array_1 = np.load(path_1)['data']
            value_array_2 = np.load(path_2)['data']

            df_extraction_1 = pd.DataFrame(np.percentile(value_array_1, [2.5, 50, 97.5], axis=0).T, columns=['lower', 'mid', 'upper'], index=future)
            df_extraction_2 = pd.DataFrame(np.percentile(value_array_2, [2.5, 50, 97.5], axis=0).T, columns=['lower', 'mid', 'upper'], index=future)

            extraction_for_stock_turnover_l = df_extraction_1.loc[start:end, 'lower'].sum()
            extraction_for_stock_turnover_m = df_extraction_1.loc[start:end, 'mid'].sum()
            extraction_for_stock_turnover_u = df_extraction_1.loc[start:end, 'upper'].sum()
            extraction_for_stock_growth_l = df_extraction_2.loc[start:end, 'lower'].sum()
            extraction_for_stock_growth_m = df_extraction_2.loc[start:end, 'mid'].sum()
            extraction_for_stock_growth_u = df_extraction_2.loc[start:end, 'upper'].sum()

            mask_1 = economy_raster == i
            mask_2 = (economy_raster == i) & (global_building_minerals_end > global_building_minerals_start)

            total_1 = np.sum(global_building_minerals_start[mask_1])
            total_2 = np.sum(global_building_minerals_end[mask_2] - global_building_minerals_start[mask_2])

            extraction_demand_raster_for_stock_turnover_l[mask_1] = 1e3 * extraction_for_stock_turnover_l * global_building_minerals_start[mask_1] / total_1
            extraction_demand_raster_for_stock_turnover_m[mask_1] = 1e3 * extraction_for_stock_turnover_m * global_building_minerals_start[mask_1] / total_1
            extraction_demand_raster_for_stock_turnover_u[mask_1] = 1e3 * extraction_for_stock_turnover_u * global_building_minerals_start[mask_1] / total_1

            extraction_demand_raster_for_stock_growth_l[mask_2] = 1e3 * extraction_for_stock_growth_l * (global_building_minerals_end[mask_2] - global_building_minerals_start[mask_2]) / total_2
            extraction_demand_raster_for_stock_growth_m[mask_2] = 1e3 * extraction_for_stock_growth_m * (global_building_minerals_end[mask_2] - global_building_minerals_start[mask_2]) / total_2
            extraction_demand_raster_for_stock_growth_u[mask_2] = 1e3 * extraction_for_stock_growth_u * (global_building_minerals_end[mask_2] - global_building_minerals_start[mask_2]) / total_2
        
    extraction_demand_raster_l = extraction_demand_raster_for_stock_turnover_l + extraction_demand_raster_for_stock_growth_l
    extraction_demand_raster_m = extraction_demand_raster_for_stock_turnover_m + extraction_demand_raster_for_stock_growth_m
    extraction_demand_raster_u = extraction_demand_raster_for_stock_turnover_u + extraction_demand_raster_for_stock_growth_u


    RESULTS = ROOT / "results"
    output_path_L = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-{start}-{end}-{scenario}-{res}-L.tif'
    output_path_M = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-{start}-{end}-{scenario}-{res}-M.tif'
    output_path_H = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-{start}-{end}-{scenario}-{res}-H.tif'

    raster_set = [extraction_demand_raster_l, extraction_demand_raster_m, extraction_demand_raster_u]
    output_path_set = [output_path_L, output_path_M, output_path_H]

    for array, output_path in zip(raster_set, output_path_set):
        with rio.open(
            output_path,
            "w",
            driver="GTiff",
            height=economy_raster_.shape[0],
            width=economy_raster_.shape[1],
            count=1,
            dtype=np.float32,
            crs=economy_raster_.crs,
            transform=economy_raster_.transform,
            compress='lzw'
        ) as dst:
            dst.write(array, 1)


In [5]:
def add_extraction_demand_together(material, scenario, res='50km'):
    RESULTS = ROOT / "results"
    country_raster_ = rio.open(RESULTS / 'country-rasters' / f'country-raster-eck4-{res}.tif')
    country_raster = country_raster_.read(1)
    total_demand_raster_L = np.zeros(shape=country_raster_.shape, dtype="float32")
    total_demand_raster_M = np.zeros(shape=country_raster_.shape, dtype="float32")
    total_demand_raster_U = np.zeros(shape=country_raster_.shape, dtype="float32")

    for start, end in zip([2023, 2026, 2031, 2036, 2041, 2046], [2025, 2030, 2035, 2040, 2045, 2050]):
        input_path_l = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-{start}-{end}-{scenario}-{res}-L.tif'
        input_path_m = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-{start}-{end}-{scenario}-{res}-M.tif'
        input_path_u = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-{start}-{end}-{scenario}-{res}-H.tif'

        total_demand_raster_L += rio.open(input_path_l).read(1)
        total_demand_raster_M += rio.open(input_path_m).read(1)
        total_demand_raster_U += rio.open(input_path_u).read(1)
    
    global_l = np.sum(total_demand_raster_L) / 1e9
    global_m = np.sum(total_demand_raster_M) / 1e9
    global_u = np.sum(total_demand_raster_U) / 1e9

    print(f'{material} ({scenario}): {global_m:.0f} ({global_l:.0f}-{global_u:.0f}) Gt')

    output_path_l = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-2023-2050-{scenario}-{res}-L.tif'
    output_path_m = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-2023-2050-{scenario}-{res}-M.tif'
    output_path_u = RESULTS / 'extraction-rasters' / f'{material}-extraction-demand-2023-2050-{scenario}-{res}-H.tif'

    for total_demand_raster, output_path in zip(
        [total_demand_raster_L, total_demand_raster_M, total_demand_raster_U], 
        [output_path_l, output_path_m, output_path_u]):
        with rio.open(
            output_path,
            "w",
            driver="GTiff",
            height=country_raster_.shape[0],
            width=country_raster_.shape[1],
            count=1,
            dtype=np.float32,
            crs=country_raster_.crs,
            transform=country_raster_.transform,
            compress='lzw'
        ) as dst:
            dst.write(total_demand_raster, 1)

In [ ]:
for material in tqdm(['sand', 'gravel', 'limestone']):
    for start, end in tqdm(zip([2023, 2026, 2031, 2036, 2041, 2046], [2025, 2030, 2035, 2040, 2045, 2050])): 
        for scenario in tqdm(['BAU', 'CE']):
            distribute_national_extraction_to_pixel(material, start, end, scenario, '50km')

In [6]:
for material in tqdm(['sand', 'gravel', 'limestone']):
    for start, end in tqdm(zip([2023, 2026, 2031, 2036, 2041, 2046], [2025, 2030, 2035, 2040, 2045, 2050])): 
        for scenario in tqdm(['MIU', 'LE', 'RE', 'RC', 'SS', 'CCR']):
            distribute_national_extraction_to_pixel(material, start, end, scenario, '50km')

  0%|          | 0/3 [00:00<?, ?it/s]












































































































































































































































































































































































































































































































































distributing of sand extraction demand in 2023-2025 of MIU: 100%|██████████| 142/142 [00:17<00:00,  8.31it/s, country=YEM]





























































































































































































































































































































In [ ]:
for material in tqdm(['sand', 'gravel', 'limestone']):
    for start, end in tqdm(zip([2023, 2026, 2031, 2036, 2041, 2046], [2025, 2030, 2035, 2040, 2045, 2050])): 
        for scenario in tqdm(['CE10', 'CE20', 'CE30', 'CE40', 'CE50', 'CE60', 'CE70', 'CE80', 'CE90']):
            distribute_national_extraction_to_pixel(material, start, end, scenario, '50km')

In [ ]:
for scenario in tqdm(['CE10', 'CE20', 'CE30', 'CE40', 'CE50', 'CE60', 'CE70', 'CE80', 'CE90']):
    add_extraction_demand_together('sand', scenario, res='50km')
    add_extraction_demand_together('gravel', scenario, res='50km')
    add_extraction_demand_together('limestone', scenario, res='50km')

In [7]:
for scenario in tqdm(['BAU', 'MIU', 'LE', 'RE', 'RC', 'SS', 'CCR', 'CE']):
    add_extraction_demand_together('sand', scenario, res='50km')
    add_extraction_demand_together('gravel', scenario, res='50km')
    add_extraction_demand_together('limestone', scenario, res='50km')

  0%|          | 0/8 [00:00<?, ?it/s]

sand (BAU): 207 (180-234) Gt
gravel (BAU): 145 (130-162) Gt


 12%|█▎        | 1/8 [00:00<00:02,  2.84it/s]

limestone (BAU): 93 (69-116) Gt
sand (MIU): 194 (168-220) Gt


 25%|██▌       | 2/8 [00:00<00:02,  2.19it/s]

...

sand (CCR): 207 (180-234) Gt
gravel (CCR): 145 (130-162) Gt


 88%|████████▊ | 7/8 [00:03<00:00,  1.77it/s]

limestone (CCR): 83 (68-97) Gt
sand (CE): 160 (139-181) Gt
gravel (CE): 118 (105-131) Gt


100%|██████████| 8/8 [00:04<00:00,  1.93it/s]

limestone (CE): 74 (60-86) Gt
